In [ ]:
# Ray remote function to process BMLL data files
import sys
import os
import ray

# Add src directory to path
current_dir = os.getcwd()
if 'data_preprocessing' in current_dir:
    src_dir = os.path.dirname(current_dir)
else:
    src_dir = os.path.join(current_dir, 'samples', 'order_flow_ray', 'src')
sys.path.append(src_dir)

from data_preprocessing.data_access.factory import DataAccessFactory

@ray.remote
def process_bmll_file(file_path: str) -> int:
    """Process BMLL file and return row count."""
    import polars as pl
    from data_preprocessing.data_access.factory import DataAccessFactory
    data_access_s3 = DataAccessFactory.create('s3', region='us-east-1')
    df = data_access_s3.read(file_path)
    return df.select(pl.count()).collect().item()

# Initialize Ray with runtime environment
ray.init(runtime_env={"working_dir": src_dir})

# Create S3 data access for listing files
data_access_s3 = DataAccessFactory.create('s3', region='us-east-1')

# List files from BMLL data
files = data_access_s3.list_files('s3://bmlldata/2024/01/02/trades/AMERICAS/')
print(f'Found {len(files)} files')
for file_path, size in files[:5]:
    print(f'{file_path} - {size:,} bytes')

# Use first file as test
test_file = files[0][0] if files else 's3://bmlldata/2024/01/02/trades/AMERICAS/trades-ARCX-20240102.parquet'

# Process file with Ray
print(f'Processing file: {test_file}')
result = ray.get(process_bmll_file.remote(test_file))
print(f'Row count: {result}')

ray.shutdown()